In [1]:
import dataclasses
import enum
import json
import re
import functools
import os

import polars as pl

In [2]:
class Arch(enum.Enum):
    neon = enum.auto()
    neon64 = enum.auto()

    @staticmethod
    def baseline() -> "Arch":
        return Arch.neon

    def is_baseline(self) -> bool:
        return self == self.baseline()


FLOAT_RE = re.compile(r"float(?P<nbits>\d+)_t")
INT_RE = re.compile(r"int(?P<nbits>\d+)_t")
UINT_RE = re.compile(r"uint(?P<nbits>\d+)_t")


class Kind(enum.Enum):
    signed_int = "i"
    unsigned_int = "u"
    float = "f"

    @staticmethod
    def parse(s: str) -> "Kind":
        if s.startswith("f"):
            return Kind.float
        elif s.startswith("u"):
            return Kind.unsigned_int
        else:  # i or s notation
            return Kind.signed_int


class PrimitiveType(enum.Enum):
    i8 = enum.auto()
    u8 = enum.auto()
    i16 = enum.auto()
    u16 = enum.auto()
    i32 = enum.auto()
    u32 = enum.auto()
    i64 = enum.auto()
    u64 = enum.auto()
    f16 = enum.auto()
    f32 = enum.auto()
    f64 = enum.auto()

    @staticmethod
    def parse(s: str) -> "PrimitiveType":
        if m:= FLOAT_RE.search(s):
            return getattr(PrimitiveType, f"f{m["nbits"]}")
        if m:= UINT_RE.search(s):
            return getattr(PrimitiveType, f"u{m["nbits"]}")
        if m:= INT_RE.search(s):
            return getattr(PrimitiveType, f"i{m["nbits"]}")

    @staticmethod
    def make_sized(kind: Kind, nbits: int) -> "PrimitiveType":
        return getattr(PrimitiveType, f"{kind.value}{nbits}")

    @property
    def kind(self) -> Kind:
        match self.name[0]:
            case "i":
                return Kind.signed_int
            case "u":
                return Kind.unsigned_int
            case "f":
                return Kind.float
            case c:
                raise ValueError(f"unknown type prefix {c!r} in {self.name}")

    @property
    def is_signed_int(self) -> bool:
        return self.kind is Kind.signed_int

    @property
    def is_unsigned_int(self) -> bool:
        return self.kind is Kind.unsigned_int

    @property
    def is_int(self) -> bool:
        return not self.is_float

    @property
    def is_float(self) -> bool:
        return self.kind is Kind.float

    @property
    def nbits(self) -> int:
        return int(self.name[1:])

    @property
    def nbytes(self) -> int:
        return self.nbits // 8

    def to_signed(self) -> "PrimitiveType":
        if self.is_unsigned_int:
            return PrimitiveType[f"i{self.nbits}"]
        return self  # already signed, or a float

    def to_unsigned(self) -> "PrimitiveType":
        if self.is_float:
            raise ValueError(f"{self.name} has no unsigned counterpart")
        if self.is_signed_int:
            return PrimitiveType[f"u{self.nbits}"]
        return self

    @property
    def type(self) -> "PrimitiveType":
        """Meant for generic code."""
        return self

    def __lt__(self, other: "PrimitiveType") -> bool:
        """Order for sorting."""
        return self.value.__lt__(other.value)


SIMD_RE = re.compile(r"(?P<type>[a-z]+)(?P<nbits>\d+)x(?P<count>\d+)_t")


@dataclasses.dataclass(frozen=True, slots=True, order=True)
class SimdType:
    type: PrimitiveType
    count: int

    @staticmethod
    def parse(s: str) -> "SimdType":
        m = SIMD_RE.match(s)
        if m is None:
            return None
        return SimdType(
            type=PrimitiveType.make_sized(kind=Kind.parse(m["type"]), nbits=m["nbits"]),
            count=m["count"]
        )

    @property
    def nbits(self) -> int:
        return self.count * self.type.nbits

    @property
    def nbytes(self) -> int:
        return self.count * self.type.nbytes        


@dataclasses.dataclass(frozen=True, slots=True)
class Intrinsic:
    name: str
    args: list[SimdType | PrimitiveType]
    ret: SimdType | PrimitiveType
    arch: Arch

    @property
    def arity(self) -> int:
        return len(self.args)

    def arg_type(self, idx: int) -> type:
        return type(self.args[idx])
        
    def ret_type(self) -> bool:
        return type(self.ret)

    def __lt__(self, other: "Intrinsic") -> bool:
        """Order for sorting."""
        return [a.type for a in self.args].__lt__([a.type for a in other.args])


SUPPORTED_PRIMITIVE_TYPE = [
    PrimitiveType.u8,
    PrimitiveType.i8,
    PrimitiveType.u16,
    PrimitiveType.i16,
    PrimitiveType.u32,
    PrimitiveType.i32,
    PrimitiveType.u64,
    PrimitiveType.i64,
    PrimitiveType.f32,
    PrimitiveType.f64,
]


@dataclasses.dataclass(frozen=True, slots=True)
class IntrinsicFamily:
    pattern: re.Pattern

    @staticmethod
    def parse_any_type(type: str) -> SimdType | PrimitiveType:
        if (t := SimdType.parse(type)) is not None:    
            return t
        elif (t := PrimitiveType.parse(type)) is not None:   
            return t
        else:
            raise ValueError(f"Unrecognized type {type}")

    def parse_instance(
        self,
        name: str,
        args: list[str],
        ret: str,
        isa: str,
        archs: list[str],
    ) -> UnaryIntrinsic | None:
        if not self.pattern.match(name):
            return None 
        ret_p = self.parse_any_type(ret)
        if ret_p.type not in SUPPORTED_PRIMITIVE_TYPE:
            return None
        args_p = [self.parse_any_type(a.split(" ")[0]) for a in args]
        if any(a.type not in SUPPORTED_PRIMITIVE_TYPE for a in args_p):
            return None

        archs = [a.strip().lower() for a in archs]
        arch = Arch.neon64
        if isa.strip().lower() == "neon" and "v7" in archs:
            arch = Arch.neon

        return Intrinsic(name=name, args=args_p, ret=ret_p, arch=arch)                

In [3]:
# TODO: generic multi types
FMT_IS_SUPPORTED = """
template<class T, class A>
XSIMD_INLINE constexpr bool {prefix}_is_supported() {{
{body}
}}
"""

FMT_INTRINSIC = """
template<class T, class A>
XSIMD_INLINE auto {prefix}_batch({args}) {{
{body}
}}
"""

@dataclasses.dataclass(frozen=True)
class XsimdIntrinsicGenerator:
    intrinsics: list[Intrinsic]

    def __post_init__(self) -> None:
        if len(self.intrinsics) == 0:
            raise ValueError("Must have at least one intrinsic")
        if any(i.arity != self.arity for i in self.intrinsics):
            raise ValueError("All intrinsics must have same arity")
        if any(i.ret_type() is not self.ret_type() for i in self.intrinsics):
            raise ValueError("All return type must have kind of type")
        for idx in range(self.arity):
            if any(i.arg_type(idx) is not self.arg_type(idx) for i in self.intrinsics):
                raise ValueError(f"All argument {idx} must have kind of type")
    
    @property
    def arity(self) -> int:
        return self.intrinsics[0].arity

    def arg_type(self, idx: int) -> type:
        return self.intrinsics[0].arg_type(idx)
        
    def ret_type(self) -> bool:
        return self.intrinsics[0].ret_type()

    @staticmethod
    def fmt_batch(type: str, arch: str | None = None, ns: str = "") -> str:
        if len(ns) > 0 and not ns.endswith("::"):
            ns = f"{ns}::"
        arch = f", {arch}" if arch is not None else ""
        return f"{ns}batch<{type}{arch}>"

    @staticmethod
    def fmt_type(type: PrimitiveType | str) -> str:
        if isinstance(type, str):
            return type
        if type.is_float:
            if type.nbits == 16:
                return "std::float16_t"  # C++23
            elif type.nbits == 32:
                return "float"
            elif type.nbits == 64:
                return "double"
        prefix = "u" if type.is_unsigned_int else ""
        return f"std::{prefix}int{type.nbits}_t"

    @staticmethod
    def fmt_arch(arch: Arch | str) -> str:
        if isinstance(arch, str):
            return arch
        return arch.name

    @classmethod
    def fmt_type_equal(cls, lhs: PrimitiveType | str, rhs: PrimitiveType | str) -> str:
        return f"is_like_v<{cls.fmt_type(lhs)}, {cls.fmt_type(rhs)}>"

    @classmethod
    def fmt_type_any(cls, lhs: PrimitiveType | str, *rhs: PrimitiveType | str) -> str:
        rhss = ", ".join(cls.fmt_type(r) for r in rhs)
        return f"is_like_any_v<{cls.fmt_type(lhs)}, {rhss}>"

    @classmethod
    def fmt_arch_compatible(cls, arch: Arch | str, base: Arch | str) -> str:
        return f"std::is_base_of_v<{cls.fmt_arch(base)}, {cls.fmt_arch(arch)}>"

    @classmethod
    def fmt_intrinsic_supported_case(
        cls, intrinsic: Intrinsic, type: str, arch: str,
    ) -> str:
        t = cls.fmt_type_equal(type, intrinsic.args[0].type)
        a = cls.fmt_arch_compatible(arch, intrinsic.arch)
        return f"if constexpr({t}) {{ return {a}; }}"

    def fmt_sig_type(self, idx: int, type: str, arch: str,) -> str:
        if all(isinstance(i.args[idx], PrimitiveType) for i in self.intrinsics):
            return type
        elif all(isinstance(i.args[idx], SimdType) for i in self.intrinsics):
            return f"batch<{type}, {arch}>"
        else:
            raise ValueError(f"Intrinsics have different parameter types in position {idx}")

    def fmt_is_supported(self) -> str:
        # Dispatching type on first arg
        lines: list[str] = []
        full_support = []
        for intrsct in self.intrinsics:
            if not intrsct.arch.is_baseline():
                lines.append(self.fmt_intrinsic_supported_case(intrsct, type="T", arch="A"))
            else:
                full_support.append(intrsct.args[0].type)
        # The ones for which a baseline intrinsic exist
        lines.append(f"return {self.fmt_type_any("T", *full_support)};")
        body = "\n".join([f"    {l}" for l in lines])
        return FMT_IS_SUPPORTED.format(prefix=self.prefix, body=body)

    def fmt_instrinsic_case(self, intrinsic: Intrinsic, type: str, arch: str, params: list[str]) -> str:
        t = self.fmt_type_equal(type, intrinsic.args[0].type)
        return f"if constexpr({t}) {{ return {intrinsic.name}({', '.join(params)}); }}"

    def fmt_instrinsic_batch(self) -> str:
        params = "abcdefghijklmnopqrstuvwxyz"[:self.arity]
        lines: list[str] = [
            self.fmt_instrinsic_case(intrsct, type="T", arch="A", params=params)
            for intrsct in self.intrinsics
        ]
        lines.append(f'{{ static_assert(false, "unsupported type for {self.prefix}"); }}')
        body = "\n    else ".join(lines)
        assrt = f'static_assert({self.prefix}_is_supported<T, A>(), "{self.prefix} unsupported");'
        body = f'    {assrt}\n    ' + body
        args = []
        for idx in range(self.arity):
            t = self.fmt_sig_type(idx, type="T", arch="A")
            p = params[idx]
            args.append(f"{t} {p}")
        return FMT_INTRINSIC.format(prefix=self.prefix, body=body, args=", ".join(args))
        
    @functools.cached_property
    def prefix(self) -> str:
        return os.path.commonprefix([i.name for i in self.intrinsics]).strip("_")

In [4]:
FILE_HEADER="""
/****************************************************************************
 * Copyright (c) xsimd contributors                                         *
 *                                                                          *
 * Distributed under the terms of the BSD 3-Clause License.                 *
 *                                                                          *
 * The full license is in the file LICENSE, distributed with this software. *
 ****************************************************************************/

#ifndef XSIMD_OVERLOAD_{arch}_HPP
#define XSIMD_OVERLOAD_{arch}_HPP

#include "../config/xsimd_macros.hpp"
#include "../types/xsimd_batch.hpp"
#include "../utils/xsimd_type_traits.hpp"

#include <type_traits>

namespace xsimd::overload {{
"""

FILE_FOOTER="""
}}  // namespace xsimd::overload

#endif  // XSIMD_OVERLOAD_{arch}_HPP
"""

@dataclasses.dataclass(frozen=True)
class XsimdFileGenerator:
    name: str
    generators: list[XsimdIntrinsicGenerator]

    def fmt(self) -> str:
        lines = [FILE_HEADER.format(arch=self.name.upper())]
        for gen in self.generators:
            lines.append(gen.fmt_is_supported())
            lines.append(gen.fmt_instrinsic_batch())
        lines.append(FILE_FOOTER.format(arch=self.name.upper()))
        return "".join(lines)

In [5]:
def load_arm_data(file: str) -> pl.DataFrame:
    raw = json.load(open(file))
    for r in raw:
          r["Arguments_Preparation"] = [
              dict(arg=k, **v) for k, v in (r.get("Arguments_Preparation", {})).items()
          ]
    df = pl.DataFrame(raw)
    return df.with_columns(pl.col("return_type").struct.unnest().alias("return_type"))

def get_intrinsic_generator(pattern: str, df: pl.DataFrame) -> XsimdIntrinsicGenerator:
    intrinsics = []
    family = IntrinsicFamily(re.compile(pattern))
    matches = df.filter(pl.col("name").str.contains(family.pattern.pattern))
    for row in matches.iter_rows(named=True):
        inst = family.parse_instance(
            name=row["name"],
            args=row["arguments"],
            ret=row["return_type"],
            isa=row["SIMD_ISA"],
            archs=row["Architectures"],
        )
        if inst is not None:
            intrinsics.append(inst)

    return XsimdIntrinsicGenerator(sorted(intrinsics))

In [21]:
# TODO how to handle int8x8_t (half batch) ? Use in combinaison with vget_low?
# TODO quad _f32_u32
# TODO vshrq_n_s64 -> macro -> immediate

df = load_arm_data("arm_intrinsics.json")
gen = XsimdFileGenerator(
    name="neon",
    generators=[
        get_intrinsic_generator(r"vget_low_[usf]\d+", df=df),
        get_intrinsic_generator(r"vget_high_[usf]\d+", df=df),
        get_intrinsic_generator(r"vdupq_n_[usf]\d+", df=df),

        get_intrinsic_generator(r"vrev64q_[usf]\d+", df=df),
        
        # get_intrinsic_generator(r"vld1q_[usf]\d+", df=df),
        # get_intrinsic_generator(r"vst1q_[usf]\d+", df=df),

        get_intrinsic_generator(r"vandq_[usf]\d+", df=df),

        # Comparison
        get_intrinsic_generator(r"vceqq_[usf]\d+", df=df),

        # Add / Sub / Neg
        get_intrinsic_generator(r"vaddq_[usf]\d+", df=df),
        get_intrinsic_generator(r"vhaddq_[usf]\d+", df=df),
        get_intrinsic_generator(r"vrhaddq_[usf]\d+", df=df),
        get_intrinsic_generator(r"vqaddq_[usf]\d+", df=df),
        get_intrinsic_generator(r"vnegq_[usf]\d+", df=df),
        get_intrinsic_generator(r"vsubq_[usf]\d+", df=df),
        get_intrinsic_generator(r"vqsubq_[usf]\d+", df=df),

        get_intrinsic_generator(r"vmull_[usf]\d+", df=df),
        get_intrinsic_generator(r"vmulq_[usf]\d+", df=df),

        # vreinterpretq
        get_intrinsic_generator(r"vreinterpretq_s8_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_u8_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_s16_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_u16_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_s32_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_u32_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_s64_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_u64_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_f32_[usf]\d+", df=df),
        get_intrinsic_generator(r"vreinterpretq_f64_[usf]\d+", df=df),
    ]
)

In [19]:
with open("include/xsimd/overload/neon.hpp", "w+") as f:
    f.write(gen.fmt())

In [23]:
df.filter(pl.col("name") == "vshrq_n_s64")

SIMD_ISA,name,arguments,return_type,Arguments_Preparation,Architectures,instructions
str,str,list[str],str,list[struct[2]],list[str],list[list[str]]
"""Neon""","""vshrq_n_s64""","[""int64x2_t a"", ""const int n""]","""int64x2_t""","[{""a"",""Vn.2D""}, {""n"",null}]","[""v7"", ""A32"", ""A64""]","[[""SSHR""]]"


In [17]:
(
    df.select("return_type")
    .filter(pl.col("return_type").str.contains("f"))
    .filter(~pl.col("return_type").str.contains("x"))
    .unique()
)

return_type
str
"""svfloat32_t"""
"""svfloat16_t"""
"""float64_t"""
"""svfloat64_t"""
"""float16_t"""
"""float32_t"""


In [30]:
df.select("Architectures").unique()

Architectures
list[str]
"[""A64""]"
"[""v7"", ""A32"", ""A64""]"
"[""A32"", ""A64""]"


In [31]:
df.select("SIMD_ISA").unique()

SIMD_ISA
str
"""SVE2"""
"""SVE"""
"""Neon"""
